<!-- thesis-review-note -->
## Çalışma Notu

- Amaç: Hazir ProsusAI/finbert modelini hedef test bolumu uzerinde baseline olarak degerlendirir.
- Girdi: Plain sentiment splitleri ve hazir FinBERT modeli.
- Çıktı ve değerlendirme: Fine-tune edilen modellerin karsilastirilacagi ilk referans performansi ve hata yonlerini verir.
- Sıra notu: Notebook numarasi deney akışındaki yerini gösterir; önceki numaralar tamamlanmadan kalıcı sonuç yorumları güncellenmemelidir.
- Sonuç güvenliği: Bu dosyadaki mevcut output hücreleri ve kalıcı sonuç dosyaları korunur.


In [1]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
from pathlib import Path
import pandas as pd

currentDir = APP_ROOT

DATA_DIR = paths.TRAINING_DATASETS_DIR 
PLAIN_PATH = paths.PLAIN_SENTIMENT_DATASET_PARQUET_PATH
PLAIN_CSV_PATH = paths.PLAIN_SENTIMENT_DATASET_CSV_PATH

if PLAIN_PATH.exists():
    plain_sentiment_df = pd.read_parquet(PLAIN_PATH)
    print("Parquet okundu:", PLAIN_PATH)
elif PLAIN_CSV_PATH.exists():
    plain_sentiment_df = pd.read_csv(PLAIN_CSV_PATH)
    print("CSV okundu:", PLAIN_CSV_PATH)
else:
    raise FileNotFoundError(
        f"plain_sentiment_df bulunamadı.\nAranan yollar:\n{PLAIN_PATH}\n{PLAIN_CSV_PATH}"
    )

print("Shape:", plain_sentiment_df.shape)
print("\nColumns:")
print(plain_sentiment_df.columns.tolist())


Parquet okundu: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\processed\training_datasets\plain_sentiment_dataset.parquet
Shape: (16777, 22)

Columns:
['sample_id', 'source_dataset', 'dataset_full_name', 'dataset_group', 'dataset_role', 'dataset_description', 'source_original_split', 'text', 'label', 'label_id', 'original_label', 'original_score', 'target_entity_based', 'usable_for_training', 'usable_for_main_evaluation', 'finbert_comparison_suitable', 'finbert_training_overlap_risk', 'finbert_comparison_note', 'label_source_type', 'row_note', 'metadata_json', 'created_at']


In [2]:
plain_sentiment_df.tail(1)

,sample_id,source_dataset,dataset_full_name,dataset_group,dataset_role,dataset_description,source_original_split,text,label,label_id,...,target_entity_based,usable_for_training,usable_for_main_evaluation,finbert_comparison_suitable,finbert_training_overlap_risk,finbert_comparison_note,label_source_type,row_note,metadata_json,created_at
16776,PLAIN_SENT_0016777,FinancialPhraseBank,Financial PhraseBank,plain_sentiment,auxiliary_training_not_main_evaluation,Finansal haber cümlelerinden oluşan düz senten...,test,Poyry 's net sales in 2007 amounted to about E...,neutral,1,...,False,True,False,False,True,ProsusAI/finbert Financial PhraseBank üzerinde...,sentence_level_class_label,Plain sentence-level financial sentiment row; ...,"{""source_original_split"": ""test"", ""phrasebank_...",2026-05-13 13:20:23


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

df = plain_sentiment_df.copy()

# ------------------------------------------------------------
# 1) Kontrol
# ------------------------------------------------------------
print("plain_sentiment_df shape:", df.shape)

print("\nSource distribution:")
print(df["source_dataset"].value_counts(dropna=False))

print("\nsource_original_split distribution:")
print(df["source_original_split"].value_counts(dropna=False))

print("\nSource x split:")
display(pd.crosstab(df["source_dataset"], df["source_original_split"]))

print("\nSource x label:")
display(pd.crosstab(df["source_dataset"], df["label"]))


# ------------------------------------------------------------
# 2) Twitter ana değerlendirme datasını al
# ------------------------------------------------------------
twitter_df = df[
    (df["source_dataset"] == "TwitterFinancialNewsSentiment") &
    (df["usable_for_main_evaluation"] == True) &
    (df["finbert_comparison_suitable"] == True)
].copy()

twitter_df = twitter_df.dropna(subset=["text", "label"]).copy()
twitter_df["label"] = twitter_df["label"].astype(str).str.lower().str.strip()

print("\nTwitter main eval shape:", twitter_df.shape)
print(twitter_df["label"].value_counts())


# ------------------------------------------------------------
# 3) Twitter içinden final_test ayır
# ------------------------------------------------------------
twitter_train_part, final_test_df = train_test_split(
    twitter_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=twitter_df["label"]
)

twitter_train_part = twitter_train_part.copy().reset_index(drop=True)
final_test_df = final_test_df.copy().reset_index(drop=True)

twitter_train_part["experiment_split"] = "train"
final_test_df["experiment_split"] = "final_test"


# ------------------------------------------------------------
# 4) PhraseBank yardımcı eğitim verisi
# ------------------------------------------------------------
phrasebank_df = df[
    (df["source_dataset"] == "FinancialPhraseBank") &
    (df["usable_for_training"] == True)
].copy()

phrasebank_df = phrasebank_df.dropna(subset=["text", "label"]).copy()
phrasebank_df["label"] = phrasebank_df["label"].astype(str).str.lower().str.strip()
phrasebank_df["experiment_split"] = "aux_train"


# ------------------------------------------------------------
# 5) Train pool oluştur
# Twitter train part + PhraseBank
# ------------------------------------------------------------
train_pool_df = pd.concat(
    [twitter_train_part, phrasebank_df],
    ignore_index=True
)

# ------------------------------------------------------------
# 6) Leakage kontrolü
# ------------------------------------------------------------
train_pool_df["text_norm_tmp"] = train_pool_df["text"].astype(str).str.lower().str.strip()
final_test_df["text_norm_tmp"] = final_test_df["text"].astype(str).str.lower().str.strip()

test_texts = set(final_test_df["text_norm_tmp"])
train_pool_df = train_pool_df[
    ~train_pool_df["text_norm_tmp"].isin(test_texts)
].copy()

overlap = set(train_pool_df["text_norm_tmp"]).intersection(set(final_test_df["text_norm_tmp"]))

train_pool_df = train_pool_df.drop(columns=["text_norm_tmp"]).reset_index(drop=True)
final_test_df = final_test_df.drop(columns=["text_norm_tmp"]).reset_index(drop=True)


# ------------------------------------------------------------
# 7) Sonuçlar
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("FINAL TEST DF")
print("=" * 80)

print("final_test_df shape:", final_test_df.shape)

print("\nFinal test source:")
print(final_test_df["source_dataset"].value_counts())

print("\nFinal test label:")
print(final_test_df["label"].value_counts())

print("\nFinal test label ratio:")
print(final_test_df["label"].value_counts(normalize=True).mul(100).round(2))


print("\n" + "=" * 80)
print("TRAIN POOL DF")
print("=" * 80)

print("train_pool_df shape:", train_pool_df.shape)

print("\nTrain pool source:")
print(train_pool_df["source_dataset"].value_counts())

print("\nTrain pool label:")
print(train_pool_df["label"].value_counts())

print("\nTrain pool label ratio:")
print(train_pool_df["label"].value_counts(normalize=True).mul(100).round(2))

print("\nTrain/Test text overlap:", len(overlap))

if len(overlap) == 0:
    print("OK: final_test_df train_pool_df içinde yok.")
else:
    print("UYARI: overlap var.")

print("\nTrain source x label:")
display(pd.crosstab(train_pool_df["source_dataset"], train_pool_df["label"]))

print("\nFinal test source x label:")
display(pd.crosstab(final_test_df["source_dataset"], final_test_df["label"]))


plain_sentiment_df shape: (16777, 22)

Source distribution:
source_dataset
TwitterFinancialNewsSentiment    11931
FinancialPhraseBank               4846
Name: count, dtype: int64

source_original_split distribution:
source_original_split
train         12643
validation     3164
test            970
Name: count, dtype: int64

Source x split:


source_original_split,test,train,validation
source_dataset,,,
FinancialPhraseBank,970,3100,776
TwitterFinancialNewsSentiment,0,9543,2388



Source x label:


label,negative,neutral,positive
source_dataset,,,
FinancialPhraseBank,604,2879,1363
TwitterFinancialNewsSentiment,1789,7744,2398



Twitter main eval shape: (11931, 22)
label
neutral     7744
positive    2398
negative    1789
Name: count, dtype: int64

FINAL TEST DF
final_test_df shape: (2387, 23)

Final test source:
source_dataset
TwitterFinancialNewsSentiment    2387
Name: count, dtype: int64

Final test label:
label
neutral     1549
positive     480
negative     358
Name: count, dtype: int64

Final test label ratio:
label
neutral     64.89
positive    20.11
negative    15.00
Name: proportion, dtype: float64

TRAIN POOL DF
train_pool_df shape: (14388, 23)

Train pool source:
source_dataset
TwitterFinancialNewsSentiment    9542
FinancialPhraseBank              4846
Name: count, dtype: int64

Train pool label:
label
neutral     9072
positive    3281
negative    2035
Name: count, dtype: int64

Train pool label ratio:
label
neutral     63.05
positive    22.80
negative    14.14
Name: proportion, dtype: float64

Train/Test text overlap: 0
OK: final_test_df train_pool_df içinde yok.

Train source x label:


label,negative,neutral,positive
source_dataset,,,
FinancialPhraseBank,604,2879,1363
TwitterFinancialNewsSentiment,1431,6193,1918



Final test source x label:


label,negative,neutral,positive
source_dataset,,,
TwitterFinancialNewsSentiment,358,1549,480


In [4]:
train_pool_df.tail(1)

,sample_id,source_dataset,dataset_full_name,dataset_group,dataset_role,dataset_description,source_original_split,text,label,label_id,...,usable_for_training,usable_for_main_evaluation,finbert_comparison_suitable,finbert_training_overlap_risk,finbert_comparison_note,label_source_type,row_note,metadata_json,created_at,experiment_split
14387,PLAIN_SENT_0016777,FinancialPhraseBank,Financial PhraseBank,plain_sentiment,auxiliary_training_not_main_evaluation,Finansal haber cümlelerinden oluşan düz senten...,test,Poyry 's net sales in 2007 amounted to about E...,neutral,1,...,True,False,False,True,ProsusAI/finbert Financial PhraseBank üzerinde...,sentence_level_class_label,Plain sentence-level financial sentiment row; ...,"{""source_original_split"": ""test"", ""phrasebank_...",2026-05-13 13:20:23,aux_train


In [5]:
final_test_df

,sample_id,source_dataset,dataset_full_name,dataset_group,dataset_role,dataset_description,source_original_split,text,label,label_id,...,usable_for_training,usable_for_main_evaluation,finbert_comparison_suitable,finbert_training_overlap_risk,finbert_comparison_note,label_source_type,row_note,metadata_json,created_at,experiment_split
0,PLAIN_SENT_0005666,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,$WMGC - Warner Music Group Begins IPO Effort. ...,neutral,1,...,True,True,True,False,ProsusAI/finbert'in bilinen fine-tuning verisi...,document_level_class_label,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test
1,PLAIN_SENT_0009294,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Biogen shares rise on patent resolution with S...,positive,2,...,True,True,True,False,ProsusAI/finbert'in bilinen fine-tuning verisi...,document_level_class_label,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test
2,PLAIN_SENT_0004337,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Considering selling your house to an iBuyer? Y...,neutral,1,...,True,True,True,False,ProsusAI/finbert'in bilinen fine-tuning verisi...,document_level_class_label,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test
3,PLAIN_SENT_0005359,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Trump's Bi-Lat Trade Strategy Failed With Chin...,neutral,1,...,True,True,True,False,ProsusAI/finbert'in bilinen fine-tuning verisi...,document_level_class_label,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test
4,PLAIN_SENT_0001323,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,"Boeing, Airbus kept in suspense over big Dubai...",neutral,1,...,True,True,True,False,ProsusAI/finbert'in bilinen fine-tuning verisi...,document_level_class_label,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2382,PLAIN_SENT_0002528,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,$HLT - Hilton Worldwide Q4 2019 Earnings Previ...,neutral,1,...,True,True,True,False,ProsusAI/finbert'in bilinen fine-tuning verisi...,document_level_class_label,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test
2383,PLAIN_SENT_0001913,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,SunPower launches 22M-share public offering,neutral,1,...,True,True,True,False,ProsusAI/finbert'in bilinen fine-tuning verisi...,document_level_class_label,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test
2384,PLAIN_SENT_0003153,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Trump Proposes Selling 15 Million Barre

In [6]:
# ============================================================
# FINBERT ZERO-SHOT EVALUATION ON final_test_df
# Model: ProsusAI/finbert
# Eğitim yok, fine-tune yok.
# Test: final_test_df
# ============================================================

import sys
import subprocess

def pip_install(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

pip_install("torch")
pip_install("transformers")
pip_install("sklearn", "scikit-learn")
pip_install("tqdm")

import torch
import pandas as pd
import numpy as np

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

try:
    from IPython.display import display
except Exception:
    display = print


# ------------------------------------------------------------
# 1) final_test_df kontrol
# ------------------------------------------------------------
if "final_test_df" not in globals():
    raise NameError("final_test_df bulunamadı. Önce final_test_df oluşturma hücresini çalıştır.")

eval_df = final_test_df.copy()

eval_df = eval_df.dropna(subset=["text", "label"]).copy()
eval_df["text"] = eval_df["text"].astype(str)
eval_df["label"] = eval_df["label"].astype(str).str.lower().str.strip()

valid_labels = ["negative", "neutral", "positive"]
eval_df = eval_df[eval_df["label"].isin(valid_labels)].copy()
eval_df = eval_df.reset_index(drop=True)

print("Eval shape:", eval_df.shape)

print("\nEval source distribution:")
print(eval_df["source_dataset"].value_counts(dropna=False))

print("\nEval label distribution:")
display(pd.DataFrame({
    "count": eval_df["label"].value_counts(),
    "ratio_%": eval_df["label"].value_counts(normalize=True).mul(100).round(2)
}))

display(eval_df[["text", "label", "source_dataset"]].head(PREVIEW_ROWS))


# ------------------------------------------------------------
# 2) FinBERT yükle
# ------------------------------------------------------------
MODEL_NAME = "ProsusAI/finbert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("\nModel id2label:")
print(model.config.id2label)


# ------------------------------------------------------------
# 3) Label normalize fonksiyonu
# ------------------------------------------------------------
def normalize_finbert_label(label):
    """
    FinBERT çıktısını bizim ortak label formatımıza çevirir:
    negative / neutral / positive
    """
    s = str(label).lower().strip()
    
    mapping = {
        "negative": "negative",
        "neutral": "neutral",
        "positive": "positive",
        "label_0": "positive",
        "label_1": "negative",
        "label_2": "neutral",
    }
    
    return mapping.get(s, s)


# ------------------------------------------------------------
# 4) Prediction
# ------------------------------------------------------------
@torch.no_grad()
def predict_finbert(texts, batch_size=32, max_length=128):
    all_preds = []
    all_confidences = []
    all_prob_negative = []
    all_prob_neutral = []
    all_prob_positive = []
    
    # id2label'dan hangi index hangi label onu bulalım
    id_to_norm_label = {}
    for idx, raw_label in model.config.id2label.items():
        id_to_norm_label[int(idx)] = normalize_finbert_label(raw_label)
    
    for i in tqdm(range(0, len(texts), batch_size), desc="FinBERT predicting"):
        batch_texts = texts[i:i + batch_size]
        
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        enc = {k: v.to(device) for k, v in enc.items()}
        
        outputs = model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_ids = torch.argmax(probs, dim=-1)
        
        probs_cpu = probs.cpu().numpy()
        pred_ids_cpu = pred_ids.cpu().numpy()
        
        for pred_id, prob_vec in zip(pred_ids_cpu, probs_cpu):
            pred_label = id_to_norm_label[int(pred_id)]
            all_preds.append(pred_label)
            all_confidences.append(float(prob_vec[int(pred_id)]))
            
            # Probları ortak sıraya çevir
            prob_map = {
                "negative": None,
                "neutral": None,
                "positive": None
            }
            
            for class_idx, p in enumerate(prob_vec):
                norm_label = id_to_norm_label[int(class_idx)]
                if norm_label in prob_map:
                    prob_map[norm_label] = float(p)
            
            all_prob_negative.append(prob_map["negative"])
            all_prob_neutral.append(prob_map["neutral"])
            all_prob_positive.append(prob_map["positive"])
    
    return (
        all_preds,
        all_confidences,
        all_prob_negative,
        all_prob_neutral,
        all_prob_positive
    )


texts = eval_df["text"].tolist()

pred_labels, confidences, prob_neg, prob_neu, prob_pos = predict_finbert(
    texts,
    batch_size=32,
    max_length=128
)

eval_df["finbert_pred"] = pred_labels
eval_df["finbert_confidence"] = confidences
eval_df["finbert_prob_negative"] = prob_neg
eval_df["finbert_prob_neutral"] = prob_neu
eval_df["finbert_prob_positive"] = prob_pos
eval_df["finbert_correct"] = eval_df["finbert_pred"] == eval_df["label"]


# ------------------------------------------------------------
# 5) Metrics
# ------------------------------------------------------------
labels_order = ["negative", "neutral", "positive"]

y_true = eval_df["label"].tolist()
y_pred = eval_df["finbert_pred"].tolist()

acc = accuracy_score(y_true, y_pred)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=labels_order,
    average="macro",
    zero_division=0
)

precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=labels_order,
    average="weighted",
    zero_division=0
)

finbert_metrics_df = pd.DataFrame([{
    "model": "ProsusAI/finbert_zero_shot",
    "test_set": "final_test_df",
    "n_eval": len(eval_df),
    "accuracy": round(acc, 4),
    "precision_macro": round(precision_macro, 4),
    "recall_macro": round(recall_macro, 4),
    "f1_macro": round(f1_macro, 4),
    "precision_weighted": round(precision_weighted, 4),
    "recall_weighted": round(recall_weighted, 4),
    "f1_weighted": round(f1_weighted, 4),
}])

print("\n" + "=" * 80)
print("FINBERT ZERO-SHOT METRICS ON final_test_df")
print("=" * 80)
display(finbert_metrics_df)

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=labels_order,
    digits=4,
    zero_division=0
))


# ------------------------------------------------------------
# 6) Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels_order
)

finbert_cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in labels_order],
    columns=[f"pred_{x}" for x in labels_order]
)

print("\nConfusion matrix:")
display(finbert_cm_df)


# ------------------------------------------------------------
# 7) Dağılımlar
# ------------------------------------------------------------
print("\nTrue label distribution:")
display(eval_df["label"].value_counts())

print("\nFinBERT prediction distribution:")
display(eval_df["finbert_pred"].value_counts())

print("\nCorrect ratio by true label:")
display(
    eval_df.groupby("label")["finbert_correct"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "accuracy_by_label"})
    .sort_index()
)


# ------------------------------------------------------------
# 8) Yanlış tahmin örnekleri
# ------------------------------------------------------------
finbert_wrong_df = eval_df[eval_df["finbert_correct"] == False].copy()

print("\nWrong predictions:", finbert_wrong_df.shape)

display(
    finbert_wrong_df[
        [
            "text",
            "label",
            "finbert_pred",
            "finbert_confidence",
            "finbert_prob_negative",
            "finbert_prob_neutral",
            "finbert_prob_positive",
            "source_dataset"
        ]
    ].head(PREVIEW_ROWS)
)

Eval shape: (2387, 23)

Eval source distribution:
source_dataset
TwitterFinancialNewsSentiment    2387
Name: count, dtype: int64

Eval label distribution:


,count,ratio_%
label,,
neutral,1549,64.89
positive,480,20.11
negative,358,15.00


,text,label,source_dataset
0,$WMGC - Warner Music Group Begins IPO Effort. ...,neutral,TwitterFinancialNewsSentiment
1,Biogen shares rise on patent resolution with S...,positive,TwitterFinancialNewsSentiment
2,Considering selling your house to an iBuyer? Y...,neutral,TwitterFinancialNewsSentiment



Device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

finbert_pred
neutral     1369
negative     552
positive     466
Name: count, dtype: int64


Correct ratio by true label:


,count,accuracy_by_label
label,,
negative,358,0.768156
neutral,1549,0.759199
positive,480,0.618750



Wrong predictions: (639, 29)


,text,label,finbert_pred,finbert_confidence,finbert_prob_negative,finbert_prob_neutral,finbert_prob_positive,source_dataset
3,Trump's Bi-Lat Trade Strategy Failed With Chin...,neutral,negative,0.660014,0.660014,0.284538,0.055449,TwitterFinancialNewsSentiment
4,"Boeing, Airbus kept in suspense over big Dubai...",neutral,negative,0.782608,0.782608,0.185187,0.032205,TwitterFinancialNewsSentiment
8,Canadian National Railway on watch amid strike,neutral,negative,0.754019,0.754019,0.215758,0.030223,TwitterFinancialNewsSentiment


In [7]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1) Kontrol
# ------------------------------------------------------------
if "finbert_wrong_df" not in globals():
    raise NameError("finbert_wrong_df bulunamadı. Önce FinBERT evaluation hücresini çalıştır.")

if "eval_df" not in globals():
    raise NameError("eval_df bulunamadı. Önce FinBERT evaluation hücresini çalıştır.")

wrong = finbert_wrong_df.copy()
all_eval = eval_df.copy()

print("Total eval:", all_eval.shape)
print("Wrong:", wrong.shape)
print("Error rate:", round(len(wrong) / len(all_eval) * 100, 2), "%")

display(wrong.head(PREVIEW_ROWS))

Total eval: (2387, 29)
Wrong: (639, 29)
Error rate: 26.77 %


,sample_id,source_dataset,dataset_full_name,dataset_group,dataset_role,dataset_description,source_original_split,text,label,label_id,...,row_note,metadata_json,created_at,experiment_split,finbert_pred,finbert_confidence,finbert_prob_negative,finbert_prob_neutral,finbert_prob_positive,finbert_correct
3,PLAIN_SENT_0005359,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Trump's Bi-Lat Trade Strategy Failed With Chin...,neutral,1,...,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test,negative,0.660014,0.660014,0.284538,0.055449,False
4,PLAIN_SENT_0001323,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,"Boeing, Airbus kept in suspense over big Dubai...",neutral,1,...,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test,negative,0.782608,0.782608,0.185187,0.032205,False
8,PLAIN_SENT_0001345,TwitterFinancialNewsSentiment,Twitter Financial News Sentiment,plain_sentiment,main_train_validation_test_candidate,Finansla ilişkili kısa metin/tweet veri setidi...,train,Canadian National Railway on watch amid strike,neutral,1,...,Plain financial sentiment row; no target/entit...,"{""source_original_split"": ""train""}",2026-05-13 13:20:23,final_test,negative,0.754019,0.754019,0.215758,0.030223,False


In [8]:
# ------------------------------------------------------------
# 2) Hata tipi: gerçek label -> FinBERT tahmini
# ------------------------------------------------------------
wrong["error_type"] = wrong["label"] + " -> " + wrong["finbert_pred"]

error_type_counts = wrong["error_type"].value_counts()

error_type_df = pd.DataFrame({
    "count": error_type_counts,
    "ratio_%": error_type_counts.div(len(wrong)).mul(100).round(2)
})

print("Error type distribution:")
display(error_type_df)

print("\nError type crosstab:")
display(pd.crosstab(wrong["label"], wrong["finbert_pred"]))

Error type distribution:


,count,ratio_%
error_type,,
neutral -> negative,223,34.90
neutral -> positive,150,23.47
positive -> neutral,129,20.19
negative -> neutral,64,10.02
positive -> negative,54,8.45
negative -> positive,19,2.97



Error type crosstab:


finbert_pred,negative,neutral,positive
label,,,
negative,0,64,19
neutral,223,0,150
positive,54,129,0


In [9]:
# ------------------------------------------------------------
# 4) Neutral olan ama negative tahmin edilenler - TAM METİN
# ------------------------------------------------------------

import pandas as pd

# Pandas görüntü ayarları
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 3000)

neutral_to_negative = wrong[
    (wrong["label"] == "neutral") &
    (wrong["finbert_pred"] == "negative")
].copy()

neutral_to_negative = neutral_to_negative.sort_values(
    "finbert_confidence",
    ascending=False
).reset_index(drop=False)

print("neutral -> negative:", neutral_to_negative.shape)

# ------------------------------------------------------------
# A) Notebook içinde tam metinli tablo
# ------------------------------------------------------------
display_cols = [
    "index",
    "text",
    "label",
    "finbert_pred",
    "finbert_confidence",
    "finbert_prob_negative",
    "finbert_prob_neutral",
    "finbert_prob_positive"
]

display(
    neutral_to_negative[display_cols]
    .head(PREVIEW_ROWS)
    .style.set_properties(subset=["text"], **{
        "white-space": "pre-wrap",
        "text-align": "left",
        "max-width": "1400px"
    })
)

# ------------------------------------------------------------
# B) En garanti çıktı: print ile tam metin
# Bunu bana direkt kopyalayıp atabilirsin.
# ------------------------------------------------------------
print("\n\n" + "=" * 120)
print("NEUTRAL -> NEGATIVE TAM METİN ÖRNEKLERİ")
print("=" * 120)

for _, row in neutral_to_negative.head(PREVIEW_ROWS).iterrows():
    print("\n" + "-" * 120)
    print(f"original_index        : {row['index']}")
    print(f"true_label            : {row['label']}")
    print(f"finbert_pred          : {row['finbert_pred']}")
    print(f"finbert_confidence    : {row['finbert_confidence']:.6f}")
    print(f"prob_negative         : {row['finbert_prob_negative']:.6f}")
    print(f"prob_neutral          : {row['finbert_prob_neutral']:.6f}")
    print(f"prob_positive         : {row['finbert_prob_positive']:.6f}")
    print("text:")
    print(row["text"])

neutral -> negative: (223, 31)


,index,text,label,finbert_pred,finbert_confidence,finbert_prob_negative,finbert_prob_neutral,finbert_prob_positive
0,2168,"Tyson Foods Q1 beef sales fell 8% to $3.84 bln, chicken sales rose 4.5% to $3.29 bln",neutral,negative,0.975574,0.975574,0.016051,0.008375
1,1555,"PNB Under-Reported Bad Loans By Rs 2,617 Crore In FY19",neutral,negative,0.974025,0.974025,0.010590,0.015385
2,1792,"U.S. stocks end higher Friday, but S&P 500 suffers weekly decline https://t.co/0BZLR3sOoF",neutral,negative,0.971648,0.971648,0.011595,0.016757




NEUTRAL -> NEGATIVE TAM METİN ÖRNEKLERİ

------------------------------------------------------------------------------------------------------------------------
original_index        : 2168
true_label            : neutral
finbert_pred          : negative
finbert_confidence    : 0.975574
prob_negative         : 0.975574
prob_neutral          : 0.016051
prob_positive         : 0.008375
text:
Tyson Foods Q1 beef sales fell 8% to $3.84 bln, chicken sales rose 4.5% to $3.29 bln

------------------------------------------------------------------------------------------------------------------------
original_index        : 1555
true_label            : neutral
finbert_pred          : negative
finbert_confidence    : 0.974025
prob_negative         : 0.974025
prob_neutral          : 0.010590
prob_positive         : 0.015385
text:
PNB Under-Reported Bad Loans By Rs 2,617 Crore In FY19

-------------------------------------------------------------------------------------------------------------

# 02 - Hazır FinBERT Baseline ve Hata Analizi

Bu bölümde, hazır `ProsusAI/finbert` modelinin gerçek etiketi `neutral` olan örnekleri `negative` olarak tahmin ettiği hata tipi incelenmiştir. Bu hata tipi önemlidir; çünkü FinBERT'in final test setindeki en dikkat çekici problemlerinden biri, `neutral` sınıfını bazı durumlarda gereğinden fazla `negative` olarak yorumlamasıdır.

## 1. Hata Tipinin Genel Özeti

İncelenen örneklerde:

- Gerçek etiket: `neutral`
- FinBERT tahmini: `negative`
- FinBERT güven skoru: çoğu örnekte oldukça yüksek

Bazı örneklerde modelin `negative` tahminine verdiği olasılık %95'in üzerindedir. Bu durum, FinBERT'in yalnızca hata yapmakla kalmadığını, aynı zamanda bu hatalarda oldukça emin olduğunu göstermektedir.

Örnekler:

| Metin | Gerçek Etiket | FinBERT Tahmini | Negative Olasılığı |
|---|---|---|---:|
| Tyson Foods Q1 beef sales fell 8% to $3.84 bln, chicken sales rose 4.5% to $3.29 bln | neutral | negative | 0.9756 |
| PNB Under-Reported Bad Loans By Rs 2,617 Crore In FY19 | neutral | negative | 0.9740 |
| U.S. stocks end higher Friday, but S&P 500 suffers weekly decline | neutral | negative | 0.9716 |
| Fed Intervenes With $45.55 Billion Weekend Repo, But Overall Liquidity Ticks Down | neutral | negative | 0.9710 |
| FDA inspections of overseas pharmaceutical manufacturers declined 10% from 2016 to 2018 | neutral | negative | 0.9697 |
| Eicher Motors Q3 Results: Profit Falls, Realisation Improves | neutral | negative | 0.9686 |
| Bristol-Myers Squibb reports 33% jump in Q4 revenue, Opdivo sales fall 2% | neutral | negative | 0.9665 |
| Averages end flat | neutral | negative | 0.9507 |
| Stocks finish mixed as Nasdaq logs third consecutive all-time closing high | neutral | negative | 0.9512 |
| BlackRock slashes stake in Peabody Energy | neutral | negative | 0.9536 |

---

## 2. FinBERT Neden Bu Örnekleri Negative Görüyor?

Bu örneklerde FinBERT'in özellikle olumsuz çağrışımlı kelimelere yüksek duyarlılık gösterdiği görülmektedir.

Sık görülen negatif ifadeler:

- `fell`
- `falls`
- `declined`
- `decline`
- `bad loans`
- `suffers`
- `profit falls`
- `sales fall`
- `slashes`
- `loses confidence`
- `strike`
- `arrests`
- `spying claims`
- `deep concerns`
- `delaying`
- `charges`
- `power outages`
- `support break`
- `missing snow`
- `scandal`

Bu kelimeler klasik haber duygu analizi açısından çoğu zaman negatif sinyal taşır. Bu nedenle FinBERT'in bu örnekleri `negative` olarak tahmin etmesi modelin kendi öğrendiği duygu mantığı açısından anlaşılabilir.

Örneğin:

- `beef sales fell 8%`
- `bad loans`
- `profit falls`
- `sales fall`
- `ratings have been sagging`
- `slashes stake`
- `loses confidence`
- `major power outages`
- `deep concerns`
- `spying scandal`

ifadeleri genel finansal haber sentiment açısından olumsuz görünmektedir.

---

## 3. Veri Seti Neden Neutral Etiketlemiş Olabilir?

Burada önemli nokta, Twitter Financial News Sentiment veri setindeki `neutral` etiketinin klasik duygu analizindeki `neutral` kavramıyla birebir aynı olmayabilmesidir.

Bu veri seti daha çok kısa finansal metinler, piyasa haberleri ve sosyal medya/ticker tarzı ifadelerden oluşmaktadır. Etiketler çoğu durumda klasik "olumlu/olumsuz duygu"dan ziyade piyasa yönü ile ilişkili görünmektedir:

- `bullish` → positive
- `bearish` → negative
- yönsüz / karışık / net sinyal yok → neutral

Bu nedenle bir metinde olumsuz kelimeler geçse bile, eğer açık bir piyasa yönü veya yatırım sinyali yoksa veri setinde `neutral` olarak etiketlenmiş olabilir.

Örnekler:

### 3.1. Karışık Bilgi İçeren Örnekler

`Tyson Foods Q1 beef sales fell 8%, chicken sales rose 4.5%`

Bu cümlede hem negatif hem pozitif bilgi vardır. Beef satışları düşerken chicken satışları artmıştır. Bu yüzden veri setinin bunu `neutral` veya mixed olarak değerlendirmesi mümkündür. FinBERT ise `sales fell` ifadesine ağırlık vererek `negative` tahmin etmiştir.

`Bristol-Myers Squibb reports 33% jump in Q4 revenue, Opdivo sales fall 2%`

Bu cümlede gelir artışı olumlu, Opdivo satış düşüşü olumsuzdur. Yani cümle karışık sinyal taşımaktadır. Veri setinde `neutral` olması makul olabilir. FinBERT ise `sales fall` kısmına ağırlık vererek `negative` tahmin etmiştir.

`U.S. stocks end higher Friday, but S&P 500 suffers weekly decline`

Bu metinde günlük kapanış olumlu, haftalık performans olumsuzdur. Bu nedenle `neutral` veya mixed etiket makul görünmektedir. FinBERT ise `suffers weekly decline` ifadesinden dolayı `negative` sınıfına gitmiştir.

---

### 3.2. Yönsüz veya Bilgilendirici Haberler

Bazı metinler olumsuz kelimeler içerse de doğrudan yatırım yönü belirtmeyebilir.

Örnekler:

- `Fed Intervenes With $45.55 Billion Weekend Repo`
- `FDA inspections ... declined 10%`
- `Atlanta Fed Business Inflation Expectations declined`
- `Averages end flat`
- `Stocks finish mixed`

Bu metinler piyasa bağlamında bilgi verici olabilir, ancak açık biçimde bullish veya bearish olmayabilir. Bu yüzden veri seti bunları `neutral` olarak etiketlemiş olabilir.

FinBERT ise `declined`, `ticks down`, `flat`, `mixed` gibi kelimeleri olumsuz haber sentiment'i olarak yorumlamıştır.

---

### 3.3. Klasik Negatif Haber Gibi Görünen Ama Veri Setinde Neutral Olanlar

Bazı örnekler ise gerçekten tartışmalıdır:

- `PNB Under-Reported Bad Loans`
- `Ratings have been sagging`
- `BlackRock slashes stake in Peabody Energy`
- `Nomura Instinet loses confidence in Extended Stay America`
- `Nestle India Accused Of Making Rs 90 Crore Undue Profit`
- `Apple has deep concerns...`
- `Saudi Arabia arrests writers...`
- `DOJ Charges 4 Chinese Military Hackers...`
- `Tightrope Thiam: comeback CEO stung by a spying scandal`

Bu örnekler klasik finansal haber sentiment açısından negatif kabul edilebilir. Dolayısıyla bu tür satırlarda veri setinde belirli miktarda **etiket gürültüsü** veya **etiket tanımı farkı** bulunması mümkündür.

Ancak bu durum veri setinin tamamen kullanılamaz olduğu anlamına gelmez. Finansal metinlerde sentiment etiketi; haberin genel tonu, hedef varlık, piyasa yönü ve etiketleme şemasına göre değişebilmektedir.

---

## 4. Temel Problem: FinBERT'in Sentiment Tanımı ile Veri Setinin Etiket Mantığı Farklı

Bu hata analizinden çıkan ana sonuç şudur:

> FinBERT, klasik finansal haber sentiment mantığına göre negatif ifadeleri güçlü biçimde yakalamaktadır. Ancak Twitter Financial News Sentiment veri setindeki `neutral` etiketi, çoğu zaman açık bir piyasa yönü taşımayan, karışık veya bilgilendirici metinleri temsil etmektedir.

Bu nedenle FinBERT bazı örneklerde kendi açısından mantıklı bir `negative` tahmini üretirken, veri setinin etiket mantığına göre bu örnekler `neutral` kabul edilmektedir.

Yani hata yalnızca modelin başarısızlığı değildir. Aynı zamanda iki farklı sentiment tanımı arasında fark vardır:

| FinBERT'in Öğrendiği Mantık | Twitter Financial News Sentiment Mantığı |
|---|---|
| Finansal haber cümlesinin genel olumlu/olumsuz tonu | Piyasa yönü / bullish-bearish-neutral sinyali |
| Olumsuz kelimelere duyarlı | Açık yön yoksa neutral olabilir |
| Financial PhraseBank tarzı cümleler | Kısa haber, ticker, sosyal medya ve piyasa başlığı tarzı metinler |

---

## 5. Bu Bulgular Tez İçin Neden Önemli?

Bu hata tipi tez çalışmasının motivasyonunu güçlendirmektedir.

Hazır FinBERT finansal alanda eğitilmiş güçlü bir modeldir. Ancak hedef veri setindeki kısa finansal metinlerde ve piyasa yönlü etiket mantığında her zaman başarılı değildir.

Özellikle şu gözlem önemlidir:

> FinBERT, gerçek etiketi `neutral` olan çok sayıda örneği yüksek güvenle `negative` olarak tahmin etmektedir.

Bu durum, hedef veri setine özgü fine-tuning yapılmasının gerekli olduğunu göstermektedir. Çünkü hedef veri setinde modelin yalnızca negatif kelimeleri yakalaması yeterli değildir; aynı zamanda karışık, yönsüz veya piyasa açısından net sinyal taşımayan metinleri `neutral` olarak ayırt edebilmesi gerekir.

---

## 6. Model Geliştirme Açısından Çıkarım

Bizim eğiteceğimiz BERT modeli, Twitter Financial News Sentiment veri setinin train kısmını göreceği için şu ayrımları öğrenme şansına sahip olacaktır:

- `fell` veya `declined` geçen her cümle otomatik olarak negative değildir.
- Hem olumlu hem olumsuz bilgi içeren cümleler neutral olabilir.
- `stocks finish mixed`, `averages end flat` gibi ifadeler neutral olarak değerlendirilmelidir.
- Piyasa yönü net olmayan haberler negative sınıfına atılmamalıdır.
- Ticker ve kısa piyasa haberleri, klasik haber cümlelerinden farklı yorumlanmalıdır.

Bu nedenle BERT modelinin hedef veri setinde fine-tune edilmesi, FinBERT'in özellikle `neutral -> negative` hatalarını azaltabilir.

---

## 7. Tezde Kullanılabilecek Sonuç Paragrafı

FinBERT hata analizi, modelin özellikle `neutral` sınıfında hedef veri setiyle uyum sorunu yaşadığını göstermektedir. Gerçek etiketi `neutral` olan birçok örnek, model tarafından yüksek güvenle `negative` olarak tahmin edilmiştir. Bu örneklerde `fell`, `declined`, `bad loans`, `profit falls`, `slashes`, `loses confidence`, `strike`, `arrests`, `deep concerns` ve `scandal` gibi olumsuz çağrışımlı ifadeler bulunmaktadır.

Bu bulgu, FinBERT'in klasik finansal haber sentiment mantığına göre olumsuz kelimeleri güçlü biçimde yakaladığını; ancak Twitter Financial News Sentiment veri setindeki `neutral` etiketinin klasik duygu analizindeki neutral kavramıyla tam olarak örtüşmediğini göstermektedir. Veri setindeki `neutral` sınıfı, çoğu zaman açık bir bullish veya bearish yön taşımayan, karışık veya bilgilendirici piyasa metinlerini temsil etmektedir.

Dolayısıyla FinBERT'in bu veri setindeki hataları yalnızca model başarısızlığı olarak değil, aynı zamanda hedef veri setinin etiket tanımı ile FinBERT'in öğrenmiş olduğu sentiment tanımı arasındaki fark olarak değerlendirilmelidir. Bu durum, hedef veri seti üzerinde BERT modelinin fine-tune edilmesini gerekli ve anlamlı kılmaktadır.

---

## 8. Genel Sonuç

Bu analiz sonucunda veri setinin tamamen problemli olmadığı, ancak etiket mantığının dikkatli yorumlanması gerektiği görülmektedir. Twitter Financial News Sentiment veri seti klasik haber sentimentinden ziyade finansal kısa metinlerde piyasa yönü ve yatırımcı algısı odaklı bir etiket yapısına sahiptir.

Bu nedenle tez kapsamında veri seti şu şekilde konumlandırılmalıdır:

> Bu çalışma, klasik finansal haber cümlelerinden ziyade kısa finansal metinlerde ve piyasa yönlü ifadelerde duygu analizi problemine odaklanmaktadır.

Bu konumlandırma ile veri seti tez için kullanılabilir durumdadır. Ayrıca FinBERT'in hata örnekleri, hedef veri setine özgü fine-tuning ihtiyacını açık biçimde göstermektedir.


In [10]:
# ------------------------------------------------------------
# 5) Zıt sınıf hataları - tam metin gösterimli
# ------------------------------------------------------------
positive_to_negative = wrong[
    (wrong["label"] == "positive") & 
    (wrong["finbert_pred"] == "negative")
].copy()

negative_to_positive = wrong[
    (wrong["label"] == "negative") & 
    (wrong["finbert_pred"] == "positive")
].copy()

print("positive -> negative:", positive_to_negative.shape)
print("negative -> positive:", negative_to_positive.shape)

print("\nPositive -> Negative örnekleri:")
display(
    positive_to_negative[
        ["text", "label", "finbert_pred", "finbert_confidence"]
    ]
    .sort_values("finbert_confidence", ascending=False)
    .head(PREVIEW_ROWS)
    .style.set_properties(subset=["text"], **{
        "white-space": "pre-wrap",
        "text-align": "left",
        "max-width": "1200px"
    })
)

print("\nNegative -> Positive örnekleri:")
display(
    negative_to_positive[
        ["text", "label", "finbert_pred", "finbert_confidence"]
    ]
    .sort_values("finbert_confidence", ascending=False)
    .head(PREVIEW_ROWS)
    .style.set_properties(subset=["text"], **{
        "white-space": "pre-wrap",
        "text-align": "left",
        "max-width": "1200px"
    })
)

positive -> negative: (54, 30)
negative -> positive: (19, 30)

Positive -> Negative örnekleri:


,text,label,finbert_pred,finbert_confidence
1644,Stock Market Update: Wholesale inventories decline in December,positive,negative,0.973877
1740,Stock Market Update: Weekly jobless claims drop below consensus,positive,negative,0.970280
147,Gold Mine Output Falls For First Time Since 2008 https://t.co/W6LYcZvzxt,positive,negative,0.965220



Negative -> Positive örnekleri:


,text,label,finbert_pred,finbert_confidence
73,Bank of America sees demand surge for paycheck protection loans https://t.co/6feALqK8zN,negative,positive,0.938881
1628,Trans Mountain expansion costs soars to C$12.6B - report,negative,positive,0.919704
1642,BJ's Wholesale narrows 2019 EPS outlook to $1.42-$1.50 from $1.44-$1.48,negative,positive,0.899043


# FinBERT Zero-Shot Sonuçları ve Hata Analizi

Bu aşamada `ProsusAI/finbert` modeli herhangi bir fine-tuning yapılmadan, yalnızca hazır haliyle test edilmiştir. Amaç, hazır FinBERT modelinin hedef veri setimiz olan **Twitter Financial News Sentiment** üzerindeki başlangıç performansını ölçmektir.

## 1. Test Veri Seti

Final karşılaştırma için kullanılan test kümesi yalnızca **Twitter Financial News Sentiment** veri setinden ayrılmıştır. Financial PhraseBank test verisi olarak kullanılmamıştır; çünkü FinBERT'in Financial PhraseBank üzerinde fine-tune edilmiş olması, bu veri üzerinde yapılacak karşılaştırmayı yanlı hale getirebilir.

| Özellik | Değer |
|---|---:|
| Test veri seti | Twitter Financial News Sentiment |
| Test örneği sayısı | 2.387 |
| Negative örnek sayısı | 358 |
| Neutral örnek sayısı | 1.549 |
| Positive örnek sayısı | 480 |

Test setinde `neutral` sınıfı baskındır. Bu nedenle yalnızca accuracy metriği yeterli değildir. Ana değerlendirme metriği olarak **macro-F1** dikkate alınmalıdır.

---

## 2. FinBERT Zero-Shot Performansı

| Model | Accuracy | Macro Precision | Macro Recall | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|---:|
| ProsusAI/finbert zero-shot | 0.7323 | 0.6648 | 0.7154 | 0.6794 | 0.7400 |

Sınıf bazında sonuçlar:

| Sınıf | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Negative | 0.4982 | 0.7682 | 0.6044 | 358 |
| Neutral | 0.8590 | 0.7592 | 0.8060 | 1549 |
| Positive | 0.6373 | 0.6188 | 0.6279 | 480 |

Bu sonuçlara göre FinBERT, hazır haliyle hedef veri setinde makul bir performans göstermektedir. Ancak özellikle `negative` sınıfında precision değerinin düşük olması dikkat çekmektedir. Model birçok örneği gereğinden fazla `negative` olarak sınıflandırmaktadır.

---

## 3. Confusion Matrix Yorumu

| Gerçek Sınıf | Negative Tahmin | Neutral Tahmin | Positive Tahmin |
|---|---:|---:|---:|
| True Negative | 275 | 64 | 19 |
| True Neutral | 223 | 1176 | 150 |
| True Positive | 54 | 129 | 297 |

Confusion matrix incelendiğinde en önemli hata tiplerinden biri şudur:

> Gerçek etiketi `neutral` olan 223 örnek, FinBERT tarafından `negative` olarak tahmin edilmiştir.

Bu durum, FinBERT'in bazı finansal kısa metinlerde olumsuz çağrışımlı kelimelere fazla duyarlı olduğunu göstermektedir.

---

## 4. Positive Etiketli Örneklerin Negative Tahmin Edilmesi

FinBERT'in yüksek güvenle yanlış tahmin ettiği bazı `positive -> negative` örnekleri şunlardır:

| Metin | Gerçek Etiket | FinBERT Tahmini | Güven |
|---|---|---|---:|
| Stock Market Update: Weekly jobless claims drop below consensus | positive | negative | 0.9703 |
| Stock Market Live Updates: Stocks reverse losses as China says 'Phase One' deal is all but done | positive | negative | 0.9379 |
| China suspends planned tariffs scheduled for December 15 on some U.S. goods | positive | negative | 0.9359 |
| Department store stocks bounce off lows | positive | negative | 0.9206 |
| Futures, Global Markets Soar For Second Day As Virus Fears Fade | positive | negative | 0.9036 |
| Nasdaq closes about 21 points, 0.2%, higher | positive | negative | 0.7384 |
| All major indexes finished the day in the green | positive | negative | 0.7299 |

Bu örneklerde FinBERT'in özellikle şu kelimelere veya ifadelere olumsuz tepki verdiği görülmektedir:

- `drop`
- `losses`
- `tariffs`
- `fears`
- `weak`
- `cuts`
- `falls`

Ancak finansal bağlamda bu ifadeler her zaman negatif değildir. Örneğin:

- `jobless claims drop` ifadesi ekonomik açıdan olumlu kabul edilebilir.
- `stocks reverse losses` ifadesi piyasaların toparlandığını gösterir.
- `tariffs suspended` piyasa açısından olumlu bir gelişme olabilir.
- `virus fears fade` risk algısının azaldığını gösterebilir.
- `indexes finished in the green` açık biçimde pozitif piyasa hareketidir.

Bu nedenle FinBERT'in bazı durumlarda yüzeysel kelime anlamına fazla odaklandığı, finansal/piyasa bağlamını yeterince yakalayamadığı görülmektedir.

---

## 5. Negative Etiketli Örneklerin Positive Tahmin Edilmesi

FinBERT'in `negative -> positive` yaptığı bazı örnekler de mevcuttur:

| Metin | Gerçek Etiket | FinBERT Tahmini | Güven |
|---|---|---|---:|
| Bank of America sees demand surge for paycheck protection loans | negative | positive | 0.9389 |
| Trans Mountain expansion costs soars to C$12.6B | negative | positive | 0.9197 |
| BJ's Wholesale narrows 2019 EPS outlook | negative | positive | 0.8990 |
| Celanese Q4 profits almost halved | negative | positive | 0.8937 |
| Egypt's inflation inched higher for a third straight month | negative | positive | 0.8621 |
| U.S. Dollar Uptick Limiting Gold’s Gains | negative | positive | 0.8039 |
| Rio Tinto downgraded to neutral from overweight | negative | positive | 0.5796 |
| Major indexes in the red after hitting record highs | negative | positive | 0.3912 |

Bu örneklerde ise FinBERT'in bazı pozitif çağrışımlı kelimelere takıldığı düşünülebilir:

- `demand surge`
- `expansion`
- `gains`
- `growth`
- `record highs`
- `sales climb`

Ancak bağlam dikkate alındığında bu örnekler çoğunlukla negatif veya temkinli piyasa sinyali taşımaktadır:

- `costs soar` maliyet artışını gösterir.
- `profits almost halved` kârın ciddi şekilde düştüğünü gösterir.
- `inflation inched higher` enflasyon baskısını gösterir.
- `downgraded to neutral from overweight` yatırım tavsiyesinde aşağı yönlü revizyon anlamına gelir.
- `indexes in the red` piyasanın negatif kapandığını gösterir.

---

## 6. Genel Değerlendirme

Bu hata analizi, veri setinin tamamen problemli olduğunu değil, FinBERT'in hedef veri setinin dil ve etiket yapısına tam uyumlu olmadığını göstermektedir.

Twitter Financial News Sentiment veri setindeki etiketler çoğu zaman klasik cümle duygu analizinden ziyade piyasa yönü, yani `bullish / bearish / neutral` mantığına yakındır. FinBERT ise daha çok finansal haber cümlelerinin genel duygu tonuna göre eğitilmiş bir modeldir. Bu nedenle iki yaklaşım arasında doğal bir fark oluşmaktadır.

Örneğin, `jobless claims drop` ifadesi genel dilde `jobless` ve `drop` gibi negatif çağrışımlı kelimeler içerirken, finansal piyasa açısından olumlu bir gelişme olarak değerlendirilebilir. Benzer şekilde `stocks reverse losses` ifadesi içerisinde `losses` kelimesi geçmesine rağmen, anlam olarak piyasadaki toparlanmayı ifade eder.

Bu nedenle hazır FinBERT'in hata yaptığı örnekler, hedef veri setine özgü fine-tuning ihtiyacını göstermektedir.

---

## 7. Tez Açısından Sonuç

Bu bulgular tez çalışması açısından şu sonucu desteklemektedir:

> Hazır FinBERT modeli finansal alan için geliştirilmiş güçlü bir başlangıç modeli olmasına rağmen, hedef veri setindeki kısa finansal metinler, ticker sembolleri ve piyasa yönlü ifadeler üzerinde sınırlı performans göstermektedir. Bu durum, genel amaçlı BERT modelinin hedef veri seti üzerinde fine-tune edilmesinin ve gerekirse yardımcı finansal veri setleriyle desteklenmesinin anlamlı bir araştırma problemi olduğunu göstermektedir.

Bu nedenle çalışmada FinBERT yeniden eğitilmeyecek; yalnızca hazır baseline model olarak kullanılacaktır. Buna karşılık `bert-base-uncased` modeli hedef veri seti üzerinde fine-tune edilerek aynı final test seti üzerinde FinBERT ile karşılaştırılacaktır.

Ana başarı metriği olarak accuracy yerine macro-F1 kullanılacaktır. Bunun nedeni test veri setinde `neutral` sınıfının baskın olmasıdır. Macro-F1, her sınıfın performansını daha dengeli değerlendirdiği için bu problemde daha uygun bir metriktir.

---

## 8. Sonraki Adım

Bir sonraki aşamada şu modeller karşılaştırılacaktır:

| Model | Eğitim | Test |
|---|---|---|
| ProsusAI/finbert | Eğitim yok | final_test_df |
| bert-base-uncased | Twitter train + yardımcı eğitim verileri | final_test_df |
| bert-base-uncased + pseudo-label | FinBERT yüksek güvenli pseudo-label veri + Twitter train | final_test_df |

Amaç, hedef veri setine fine-tune edilen BERT modelinin hazır FinBERT baseline'ına göre özellikle macro-F1 metriğinde daha iyi sonuç verip vermediğini incelemektir.